# 0. 구글 드라이브 연결

In [ ]:
## -----------------------------------------------------------------------------
## 반드시 본인의 구글 드라이브 위치로 수정할 것
## -----------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

root_dir = '/content/drive/MyDrive/26-1_강화학습/기말프로젝트'
filepath_smiles_list = root_dir + '/' + '250k_rndm_zinc_drugs_clean.csv'

Mounted at /content/drive


# 1. 환경 설정 (수정 금지)

In [ ]:
## rdkit 라이브러리 설치
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 49.2 MB/s eta 0:00:00


In [ ]:
## rdkit 설치 확인 및 로그 설정
import rdkit
rdkit.RDLogger.DisableLog('rdApp.*')  # Suppress RDKit warnings

## random seed 고정
import random
import numpy as np
import torch

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 주요 라이브러리 버전 확인
print(torch.__version__)
print(np.__version__)
print(rdkit.__version__)

2.11.0+cu128
2.0.2
2026.03.3


# 2. 문제 정의 (수정 금지)

In [ ]:
import pandas as pd

## Vocabulart 정의
class Vocab:
    def __init__(self, filepath: str):
        ## Special tokens
        self.START_TOK = '<START>'
        self.END_TOK = '<END>'

        ## All tokens
        self.tokens = [
            self.START_TOK, # 0. 시퀀스 시작 토큰
            self.END_TOK,  # 1. 시퀀스 종료 토큰
        ] + self._extract_vocab(filepath) # 2. SMILES 토큰 모음

        ## Token to index, index to token
        self.token_to_idx = {t:i for i, t in enumerate(self.tokens)}
        self.idx_to_token = {i:t for t, i in self.token_to_idx.items()}

        ## Special indices
        self.START_IDX = self.token_to_idx[self.START_TOK]
        self.END_IDX = self.token_to_idx[self.END_TOK]

    def _extract_vocab(self, filepath: str) -> list[str]:
        ## Load file
        df = pd.read_csv(filepath)
        smiles_list = [smi.rstrip() for smi in df['smiles'].tolist()]

        ## Extract tokens
        tokens = set()
        for smi in smiles_list:
            for token in smi:
                tokens.add(token)
        return sorted(list(tokens))

    @property
    def size(self) -> int:
        return len(self.tokens)

    def t2i(self, x: str) -> int:
        return self.token_to_idx[x]

    def i2t(self, x: int) -> str:
        return self.idx_to_token[x]

    def encode(self, smi: str) -> list[int]:
        processed_smi = smi

        # 시퀀스 시작 토큰이 이미 있으면 제거
        if processed_smi.startswith(self.START_TOK):
            processed_smi = processed_smi[len(self.START_TOK):]

        # 시퀀스 종료 토큰이 이미 있으면 제거
        if processed_smi.endswith(self.END_TOK):
            processed_smi = processed_smi[:-len(self.END_TOK)]

        # 인덱스 시퀀스 생성
        return [self.START_IDX] + [self.t2i(token) for token in processed_smi] + [self.END_IDX]

    def decode(self, indices: list[int], include_special_tokens: bool = False) -> str:
        tokens = []
        for idx in indices:
            token = self.i2t(idx)
            if token == self.END_TOK:
                if include_special_tokens:
                    tokens.append(token)
                break # 종료 토큰을 만나면 디코딩 중지
            if token == self.START_TOK:
                if include_special_tokens:
                    tokens.append(token)
            else:
                tokens.append(token)
        return ''.join(tokens)

In [ ]:
## Vocabulary Building
vocab = Vocab(filepath_smiles_list)

## Vocabulary Info.
print(vocab.tokens)
print(vocab.size)

## Examples
print(vocab.encode('CC(=O)OC1=CC=CC=C1C(=O)O'))
print(vocab.encode('<START>CC(=O)OC1=CC=CC=C1C(=O)O<END>'))
print(vocab.decode([0, 19, 19, 3, 16, 24, 4, 24, 19, 8, 16, 19, 19, 16, 19, 19, 16, 19, 8, 19, 3, 16, 24, 4, 24, 1]))
print(vocab.decode([0, 19, 19, 3, 16, 24, 4, 24, 19, 8, 16, 19, 19, 16, 19, 19, 16, 19, 8, 19, 3, 16, 24, 4, 24, 1], True))

['<START>', '<END>', '#', '(', ')', '+', '-', '/', '1', '2', '3', '4', '5', '6', '7', '8', '=', '@', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', '\\', ']', 'c', 'l', 'n', 'o', 'r', 's']
36
[0, 19, 19, 3, 16, 24, 4, 24, 19, 8, 16, 19, 19, 16, 19, 19, 16, 19, 8, 19, 3, 16, 24, 4, 24, 1]
[0, 19, 19, 3, 16, 24, 4, 24, 19, 8, 16, 19, 19, 16, 19, 19, 16, 19, 8, 19, 3, 16, 24, 4, 24, 1]
CC(=O)OC1=CC=CC=C1C(=O)O
<START>CC(=O)OC1=CC=CC=C1C(=O)O<END>


# 3. 평가 함수 (수정 금지)

In [ ]:
import sys; sys.path.append(root_dir)
from utils import penalized_logp_from_smiles, Evaluator

In [ ]:
## Examples
example_smiles_list = [
    'CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1',
    'C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1',
    'N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)cc2)cc1',
    'CCOC(=O)[C@@H]1CCCN(C(=O)c2nc(-c3ccc(C)cc3)n3c2CCCCC3)C1',
    'N#CC1=C(SCC(=O)Nc2cccc(Cl)c2)N=C([O-])[C@H](C#N)C12CCCCC2',
    'CC[NH+](CC)[C@](C)(CC)[C@H](O)c1cscc1Br',
    'COc1ccc(C(=O)N(C)[C@@H](C)C/C(N)=N/O)cc1O',
    'O=C(Nc1nc[nH]n1)c1cccnc1Nc1cccc(F)c1',
    'Cc1c(/C=N/c2cc(Br)ccn2)c(O)n2c(nc3ccccc32)c1C#N',
    'C[C@@H]1CN(C(=O)c2cc(Br)cn2C)CC[C@H]1[NH3+]',
]
print(Evaluator.evaluate(example_smiles_list * 10)) # 100개의 SMILES 문자열 입력 시

{'n_total': 100, 'n_valid': 100, 'n_unique': 10, 'validity': 1.0, 'diversity': 0.8088416928682088, 'plogp': -0.15649866696451045, 'plogp_sig': 0.48191033238674413, 'score': 0.38978916905837535}


# 4. 환경(Env) 클래스 구현

In [ ]:
class Env:
    """
    SMILES 생성을 위한 강화학습 환경 (MDP)

    - State  : 현재 스텝에서 마지막으로 생성된 토큰의 인덱스
    - Action : 다음 토큰 인덱스 (START 제외)
    - Reward : 에피소드 종료 시에만 부여
                  유효한 SMILES → penalized logP 값
                  유효하지 않음 → 0.0
    - Done   : END 토큰 선택 또는 최대 길이(100) 도달 시 True
    """

    MAX_LEN = 100


    def __init__(self, vocab: Vocab):
        self.vocab = vocab


    def reset(self):
        """
        에피소드를 초기화하고 초기 상태(START 토큰 인덱스)를 반환한다.

        Returns:
            int: START 토큰의 인덱스 (초기 상태)
        """
        self.token_ids = [self.vocab.START_IDX]
        self.done = False
        return self.vocab.START_IDX


    def step(self, action: int):
        """
        action(토큰 인덱스)을 받아 한 스텝을 진행한다.

        Args:
            action (int): 선택된 토큰 인덱스

        Returns:
            next_state (int)  : 선택된 토큰 인덱스 (다음 상태)
            reward    (float) : 즉각 보상 (종료 시에만 non-zero)
            done      (bool)  : 에피소드 종료 여부
            info      (dict)  : 종료 시 SMILES 및 pLogP 포함
        """

        assert not self.done, "이미 종료된 에피소드입니다. reset()을 먼저 호출하세요."

        self.token_ids.append(action)
        ascii_len = len(self.token_ids) - 1  # START 토큰 제외

        if action == self.vocab.END_IDX or ascii_len >= self.MAX_LEN:
            self.done = True
            smiles = self.vocab.decode(self.token_ids)
            valid = Evaluator.is_valid_smiles(smiles)
            plogp = penalized_logp_from_smiles(smiles) if valid else 0.0
            reward = plogp if valid else 0.0
            return action, reward, True, {'smiles': smiles, 'plogp': plogp, 'valid': valid}

        return action, 0.0, False, {}


## Examples
env = Env(vocab)
state = env.reset()
print(f"초기 상태(START idx): {state}")

# 유효한 SMILES 예시: CC(=O)Nc1ccc(O)cc1 (아세트아미노펜 유사)
for action in vocab.encode('CC(=O)O')[1:]:  # START 제외
    ns, reward, done, info = env.step(action)
    if done:
        print(f"SMILES: {info['smiles']}, pLogP: {info['plogp']:.4f}, valid: {info['valid']}")

초기 상태(START idx): 0
SMILES: CC(=O)O, pLogP: -1.4240, valid: True


# 5. 기타 등등

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
import pandas as pd
df = pd.read_csv(filepath_smiles_list)
print(len(df))
print(df.head())

249455
                                              smiles     logP       qed  \
0            CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1  5.05060  0.702012   
1       C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1  3.11370  0.928975   
2  N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)...  4.96778  0.599682   
3  CCOC(=O)[C@@H]1CCCN(C(=O)c2nc(-c3ccc(C)cc3)n3c...  4.00022  0.690944   
4  N#CC1=C(SCC(=O)Nc2cccc(Cl)c2)N=C([O-])[C@H](C#...  3.60956  0.789027   

        SAS  
0  2.084095  
1  3.432004  
2  2.470633  
3  2.822753  
4  4.035182  


# 5-1. Actor-Critic 네트워크 (GRU 기반)

In [ ]:
"""
에이전트의 신경망인 PolicyNet은 임베딩 층, GRU 층, 그리고 출력 선형 층으로 구성
"""
class PolicyNet(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)  # 고유 문자들을 128차원 밀집 벡터로 변환
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)  # GRU 레이어
        self.actor_head = nn.Linear(hidden_dim, vocab_size) # 최종 선형 레이어
        self.hidden_dim = hidden_dim

    def forward_seq(self, seq_batch):
        """사전학습용: 배치 처리"""
        x = self.embed(seq_batch[:, :-1])   # [B, T-1, E]
        out, _ = self.gru(x)                # [B, T-1, H]
        return self.actor_head(out)         # [B, T-1, V]

    def step(self, token_idx, h):
        """생성/RL용: 토큰 하나씩"""
        x = self.embed(torch.tensor([[token_idx]]))  # [1,1,E]
        out, h = self.gru(x, h)                      # [1,1,H]
        logits = self.actor_head(out.squeeze(0))     # [1,V]
        return logits, h

    def init_hidden(self):
        """기억 공간 초기화"""
        return torch.zeros(1, 1, self.hidden_dim)

# 5-2. 사전학습


In [ ]:
"""
pretrain 함수는 ZINC 데이터를 불러와 미니배치로 쪼갠 뒤,
문장의 길이를 맞춰주는 패딩 처리를 수행합니다.
이후 모델이 문법을 익히도록 CrossEntropy 손실 함수를 통해 가중치를 학습시키며,
RNN 계열의 고질적 문제인 그라디언트 폭발을 막기 위해 '그라디언트 클리핑' 기법을 적용했습니다.

_gen 함수는 학습된 가중치를 바탕으로 분자를 자동 생성하는 함수입니다.
문맥 기억장치(h)와 직전 토큰을 주입받아 확률 분포를 계산하며,
temperature 하이퍼파라미터를 통해 탐색의 다양성을 제어합니다.
최종적으로 확률적 샘플링을 통해 코드가 완결되면 문자열로 디코딩하여 유효한 분자식을 출력
"""
def pretrain(policy, filepath, vocab, n_samples=250000,
             epochs=3, batch_size=512, lr=1e-3):
    import pandas as pd

    ## 데이터 로드 및 토큰화 (전처리)
    df = pd.read_csv(filepath)
    all_sm = [s.rstrip() for s in df['smiles'].tolist()]
    random.shuffle(all_sm)
    train = all_sm[:n_samples]

    encoded = []
    for smi in train:
        try:
            encoded.append(torch.tensor(vocab.encode(smi))) # 문자열을 숫자로 변환
        except KeyError:
            pass  # 사전에 없는 문자면 패스

    optimizer = optim.Adam(policy.parameters(), lr=lr)

    print("=" * 55)
    print(f"사전학습: {len(encoded)}개 SMILES, {epochs} epochs")
    print("=" * 55)

    policy.train()
    for epoch in range(1, epochs + 1):
        random.shuffle(encoded)
        total_loss = 0.0; n_batches = 0

        for i in range(0, len(encoded), batch_size):
            batch = encoded[i : i + batch_size]
            max_len = max(len(s) for s in batch)  # 배치 내 최장 분자 길이 계산
            padded = torch.zeros(len(batch), max_len, dtype=torch.long) # 빈 텐서 생성
            for j, s in enumerate(batch):
                padded[j, :len(s)] = s

            logits = policy.forward_seq(padded)  # 예측값 [B, T-1, V]
            target = padded[:, 1:]               # 실제 정답 [B, T-1]
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.reshape(-1, V),  # 2차원으로 펼치기
                target.reshape(-1),     # 1차원으로 펼치기
                ignore_index=0
            )
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item(); n_batches += 1

        ## validity 샘플 체크
        policy.eval()
        v = sum(1 for _ in range(100)
                if Evaluator.is_valid_smiles(_gen(policy, vocab)))
        policy.train()
        print(f"  Epoch {epoch}/{epochs} | Loss: {total_loss/n_batches:.4f} "
              f"| Validity: {v}/100")

    print("사전학습 완료!\n")

## 한 토큰씩 분자 생성
def _gen(policy, vocab, temperature=1.0):
    ids = [vocab.START_IDX]; h = policy.init_hidden()
    with torch.no_grad():
        for _ in range(101):
            logits, h = policy.step(ids[-1], h)
            probs = F.softmax(logits / temperature, dim=-1).clone()
            probs[0][vocab.START_IDX] = 0.0
            probs = probs / probs.sum()
            a = torch.multinomial(probs, 1).item()
            ids.append(a)
            if a == vocab.END_IDX or len(ids) - 1 >= 100:
                break
    return vocab.decode(ids)

# 5-3. 에피소드 생성 (Reinforce)

In [ ]:
## 에피소드 시작 및 초기화
def generate_episode(policy, env, temperature=1.0):
    state = env.reset(); h = policy.init_hidden()
    log_probs = []; smiles = ''

    ## 토큰 생성 및 확률 분포 생성
    for _ in range(Env.MAX_LEN + 1):
        # 현재 상태와 이전 기억을 바탕으로 다음 글자 점수 계산
        logits, h = policy.step(state, h)
        probs = F.softmax(logits / temperature, dim=-1).clone()
        mask = torch.ones(vocab.size); mask[vocab.START_IDX] = 0.0
        probs = probs * mask; probs = probs / probs.sum()

        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))

        state, reward, done, info = env.step(action.item())
        if done:
            smiles = info['smiles']
            return log_probs, reward, smiles

    return log_probs, 0.0, smiles

# 5-4. Reinforce 업데이트

In [ ]:
"""
이 함수는 강화학습 단계에서 에이전트가 분자를 한 자씩 생성하며
환경과 소통하는 핵심 MDP 루프를 구현한 generate_episode 함수입니다.

에이전트는 환경으로부터 현재 상태를 받아 다음 토큰에 대한 확률 분포를 계산하고,
Categorical 분포를 통해 확률적 행동인 action을 샘플링합니다.
이때 정책 경사(Policy Gradient) 업데이트에 활용할 로그 확률값을 log_probs 리스트에
지속적으로 누적
"""
def reinforce_update(policy, optimizer, log_probs, shaped_reward, baseline=0.0):
    advantage = shaped_reward - baseline
    loss = torch.stack([-lp * advantage for lp in log_probs]).sum()
    optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
    optimizer.step()

# 5-5. 학습 실행

In [ ]:
"""
2단계 학습 전략을 취하고 있습니다.
먼저 Step 1에서 25만 개의 대규모 데이터셋과 512의 대형 배치 사이즈를 활용해
3 에포크 동안 분자의 기본 문법을 학습

이후 Step 2에서 REINFORCE 파인튜닝을 10,000 에피소드 동안 진행합니다.
이때 사전학습된 문법 지식을 파괴하지 않기 위해 3e-5라는 매우 신중한 학습률(Learning Rate)을 적용했습니다.
루프 내부에서는 에이전트가 생성한 분자가 유효하고 처음 발견된 경우
물성 점수(pLogP)에 1.5배의 가중치를 부여하는 Reward Shaping 기술을 도입하여
모델이 유효성과 다양성을 동시에 확보하며 고득점 분자를 탐색하도록 최적화
"""
PRETRAIN_SAMPLES = 250000   # 데이터셋 전체 사용
PRETRAIN_EPOCHS = 3     # 에포크
PRETRAIN_BATCH = 512    # 배치사이즈
RL_EPISODES = 10000
LEARNING_RATE = 3e-5   # 사전학습보다 낮게 (가중치 보존)
TRAIN_TEMP = 1.0      # 확률 분포의 무작위성
LOG_INTERVAL = 500    # 출력 주기

policy = PolicyNet(vocab.size)
env = Env(vocab)

## Step 1: 사전학습
pretrain(policy, filepath_smiles_list, vocab,
         n_samples=PRETRAIN_SAMPLES,
         epochs=PRETRAIN_EPOCHS,
         batch_size=PRETRAIN_BATCH)

## Step 2: REINFORCE 파인튜닝
## 사전학습보다 낮은 lr로 가중치 보존
rl_optimizer = optim.Adam(policy.parameters(), lr=LEARNING_RATE)
seen = set()
plogp_history = []

print("=" * 55)
print("REINFORCE 파인튜닝 시작 (pLogP 최대화)")
print("=" * 55)

policy.train()
for ep in range(1, RL_EPISODES + 1):
    log_probs, raw_reward, smi = generate_episode(policy, env, TRAIN_TEMP)
    is_valid = Evaluator.is_valid_smiles(smi)

    ## Reward Shaping
    if is_valid and raw_reward > -99:
        shaped = raw_reward * 1.5 if smi not in seen else 0.0
        seen.add(smi)
    else:
        shaped = 0.0   # 무효는 그냥 0 (패널티 없음 - 가중치 보존)

    reinforce_update(policy, rl_optimizer, log_probs, shaped)

    if is_valid:
        plogp_history.append(raw_reward)

    if ep % LOG_INTERVAL == 0:
        avg = np.mean(plogp_history[-200:]) if plogp_history else 0.0
        vrate = len(plogp_history) / ep
        print(f"Ep {ep:>5d} | Valid: {vrate:.3f} | "
              f"Avg pLogP: {avg:.3f} | Unique: {len(seen):>4d} | "
              f"Last: {smi[:40]}")

print("\nREINFORCE 파인튜닝 완료")

사전학습: 249455개 SMILES, 3 epochs
  Epoch 1/3 | Loss: 0.9006 | Validity: 49/100
  Epoch 2/3 | Loss: 0.6496 | Validity: 68/100
  Epoch 3/3 | Loss: 0.6089 | Validity: 70/100
사전학습 완료!

REINFORCE 파인튜닝 시작 (pLogP 최대화)
Ep   500 | Valid: 0.748 | Avg pLogP: 1.795 | Unique:  373 | Last: CCCCCCC(=O)c1ccccc1Cl
Ep  1000 | Valid: 0.839 | Avg pLogP: 2.201 | Unique:  810 | Last: C/C(=C/c1ccc(Br)s1)c1ccccc1Cl
Ep  1500 | Valid: 0.883 | Avg pLogP: 1.788 | Unique: 1187 | Last: O=C(Cc1cccc(Cl)c1I)c1ccccc1F
Ep  2000 | Valid: 0.902 | Avg pLogP: 1.490 | Unique: 1598 | Last: CC(C)c1cc(F)cc(Br)c1
Ep  2500 | Valid: 0.911 | Avg pLogP: 3.083 | Unique: 2015 | Last: CCc1cc(Cl)cc(Cl)c1C(=O)N(CC(=O)Nc1ccc(Br
Ep  3000 | Valid: 0.919 | Avg pLogP: 2.687 | Unique: 2425 | Last: CCc1cccc(Br)c1Br
Ep  3500 | Valid: 0.927 | Avg pLogP: 2.069 | Unique: 2768 | Last: Nc1cc(F)ccc1Br
Ep  4000 | Valid: 0.931 | Avg pLogP: 3.245 | Unique: 3145 | Last: O=C(CSc1cccc(Cl)c1)c1ccc(Br)cc1Cl
Ep  4500 | Valid: 0.932 | Avg pLogP: 4.507 | Unique: 3

# 5-6. 생성 및 평가

In [ ]:
"""모든 학습이 완료된 후, 모델을 eval() 모드로 전환하고
오차 역전파를 차단하여 추론 속도를 최적화했습니다.
그 후 지정된 샘플 개수만큼 분자를 독립적으로 자동 생성했습니다.

특히, 최종 평가 단계에서 GENERATE_TEMP를 1.0으로 하향 조정하여 모델이 무작위 탐색을 줄이고,
파인튜닝 과정에서 학습한 '고득점 물성 패턴'을 더 확신 있게 생성하도록 유도했습니다.
그 결과, 생성된 분자들의 화학적 유효성(Validity)이 크게 안정화되었으며,
최종 평가 모듈을 통해 산출된 종합 Score와 상위 샘플들의 유효성 여부(✓ 마크)를
통해 본 강화학습 모델이 목표 물성에 맞는 유효한 신규 분자를 성공적으로 설계해 냄을 검증"""

GENERATE_TEMP = 1.05

policy.eval()
generated_smiles = []
with torch.no_grad():
    for _ in range(Evaluator.total_samples):
        _, _, smi = generate_episode(policy, env, temperature=GENERATE_TEMP)
        generated_smiles.append(smi)

result = Evaluator.evaluate(generated_smiles)
print("\n" + "=" * 55)
print("최종 평가 결과")
print("=" * 55)
for k, v in result.items():
    print(f"  {k:>12s}: {v}")

print("\n생성된 SMILES 샘플 (처음 10개):")
for i, smi in enumerate(generated_smiles[:10]):
    mark = "✓" if Evaluator.is_valid_smiles(smi) else "✗"
    print(f"  {i+1:>2d}. [{mark}] {smi}")


최종 평가 결과
       n_total: 100
       n_valid: 100
      n_unique: 95
      validity: 1.0
     diversity: 0.7221869011558485
         plogp: 4.378708799333268
     plogp_sig: 0.8877407024080486
         score: 0.6411147069019849

생성된 SMILES 샘플 (처음 10개):
   1. [✓] C[C@@H](c1ccc(Cl)cc1Cl)c1ccc(Cl)cc1Br
   2. [✓] OC[C@@H](c1ccc(Br)cc1C(F)F)c1ccc(C(=O)Oc2ccccc2Cl)cc1
   3. [✓] O[C@H](c1cc(Cl)cc(Cl)c1Cl)c1ccc(Cl)cc1Cl
   4. [✓] CC(C)c1cc(Cl)ccc1-c1cc(Cl)ccc1Br
   5. [✓] CCC[C@H](c1ccc(Br)cc1-c1cccc(Cl)c1Cl)c1ncc(Cl)cc1Br
   6. [✓] CCCc1cc(Br)ccc1C(=O)c1ccc(Br)cc1C(=O)c1ccccc1Cl
   7. [✓] CCc1cc(C)cc(C(=O)c2ccc(C)cc2Cl)c1
   8. [✓] O[C@@H](c1ccc(Cl)cc1-c1ccccc1Cl)c1cccc(Cl)c1
   9. [✓] CCCCc1cc(Cl)cc(Br)c1-c1cc(C)cc(C)c1Cl
  10. [✓] CCCc1cc(Cl)ccc1-c1cc(Cl)ccc1-c1ccccc1Cl


In [ ]:
"""
reinforce
에피소드 종료 시에만 보상이 나오는 '희소 보상(Sparse Reward)' 환경
가벼운 연산량
사전학습 지식의 보존 유리함

ppo
스텝마다 상태 가치 예측


"""
## CSV 저장

import csv
with open('generated_smiles.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['smiles'])
    for smi in generated_smiles:
        writer.writerow([smi])
print("\ngenerated_smiles.csv 저장 완료")


generated_smiles.csv 저장 완료
